In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

def get_distinct_colors():
    """
    Returns a list of 12 distinct colors that are visually distinguishable
    """
    return [
        '#1f77b4',  # Blue
        '#ff7f0e',  # Orange
        '#2ca02c',  # Green
        '#d62728',  # Red
        '#9467bd',  # Purple
        '#8c564b',  # Brown
        '#e377c2',  # Pink
        '#7f7f7f',  # Gray
        '#bcbd22',  # Yellow-green
        '#17becf',  # Cyan
        '#ff9896',  # Light red
        '#98df8a'   # Light green
    ]

def analyze_regression_models(df):
    """
    Comprehensive analysis of regression model performances across different datasets.
    """
    # Calculate average performance for each model
    model_columns = df.columns.drop('source')
    model_means = df[model_columns].mean().sort_values()
    model_stds = df[model_columns].std()
    
    # Create performance summary
    performance_summary = pd.DataFrame({
        'Mean RMSE': model_means,
        'Std RMSE': model_stds,
        'Relative to Best': model_means - model_means.min()
    }).round(4)
    
    plt.style.use('classic')
    colors = get_distinct_colors()
    
    # 1. Grouped bar plot for comparing models across sources
    plt.figure(figsize=(20, 8)) 
    melted_df = df.melt(id_vars=['source'], var_name='Model', value_name='RMSE')
    
    # Calculate the number of models and sources for spacing
    n_models = len(model_columns)
    n_sources = len(df['source'])
    
    # Create bar positions with more space between groups
    source_positions = np.arange(n_sources) * 2
    bar_width = 1.6 / n_models  # Increased bar width
    
    # Create bars for each model with specific colors and no edges
    for i, (model, color) in enumerate(zip(model_columns, colors)):
        position = source_positions + (i - n_models/2 + 0.5) * bar_width
        plt.bar(position, df[model], width=bar_width, label=model, 
                color=color, linewidth=0)  # Removed edges by setting linewidth=0
    
    plt.xlabel('Source')
    plt.ylabel('RMSE')
    plt.title('Model Performance Comparison by Source')
    plt.xticks(source_positions, df['source'], rotation=45, ha='right')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', ncol=1)
    plt.tight_layout()
    
    # 2. Box plot of model performances with consistent colors
    plt.figure(figsize=(15, 6))
    sns.boxplot(data=melted_df, x='Model', y='RMSE', 
                palette=dict(zip(model_columns, colors)))
    plt.xticks(rotation=45, ha='right')
    plt.title('Distribution of RMSE Scores Across Models')
    plt.tight_layout()
    
    # 3. Heatmap of model correlations
    plt.figure(figsize=(12, 10))
    correlation_matrix = df[model_columns].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0)
    plt.title('Correlation Between Model Performances')
    plt.tight_layout()
    
    # 4. Performance variation across datasets with consistent colors
    plt.figure(figsize=(20, 6))
    df_normalized = df.copy()
    df_normalized[model_columns] = df_normalized[model_columns].apply(
        lambda x: (x - x.mean()) / x.std()
    )
    
    # Using the same colors for the line plot
    palette = dict(zip(model_columns, colors))
    sns.lineplot(data=df_normalized.melt(id_vars=['source'], 
                                       var_name='Model', 
                                       value_name='Normalized RMSE'),
                 x='source', y='Normalized RMSE', hue='Model',
                 palette=palette)
    plt.xticks(rotation=45, ha='right')
    plt.title('Normalized Performance Across Datasets')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()
    
    # Statistical analysis
    best_model = model_means.index[0]
    statistical_tests = {}
    for model in model_columns:
        if model != best_model:
            t_stat, p_value = stats.ttest_rel(df[best_model], df[model])
            statistical_tests[model] = {'t_statistic': t_stat, 'p_value': p_value}
    
    # Calculate win/loss ratio for each model
    win_loss = {}
    for model in model_columns:
        wins = sum((df[model] < df[other_model]).sum() 
                  for other_model in model_columns if other_model != model)
        total_comparisons = len(model_columns) - 1
        win_loss[model] = wins / (len(df) * total_comparisons)
    
    return {
        'performance_summary': performance_summary,
        'statistical_tests': pd.DataFrame(statistical_tests).T.round(4),
        'win_ratio': pd.Series(win_loss)
    }

df = pd.read_csv('rmse_imputacao_resultados_bbr.csv')
results = analyze_regression_models(df)
print("\nPerformance Summary:")
print(results['performance_summary'])
print("\nStatistical Tests vs Best Model:")
print(results['statistical_tests'])
print("\nWin Ratio:")
print(results['win_ratio'].sort_values(ascending=False))

In [ ]:
df = pd.read_csv('rmse_imputacao_resultados_cubic.csv')
results = analyze_regression_models(df)
print("\nPerformance Summary:")
print(results['performance_summary'])
print("\nStatistical Tests vs Best Model:")
print(results['statistical_tests'])
print("\nWin Ratio:")
print(results['win_ratio'].sort_values(ascending=False))

In [ ]:
df = pd.read_csv('rmse_prediction_results_bbr.csv')
results = analyze_regression_models(df)
print("\nPerformance Summary:")
print(results['performance_summary'])
print("\nStatistical Tests vs Best Model:")
print(results['statistical_tests'])
print("\nWin Ratio:")
print(results['win_ratio'].sort_values(ascending=False))

In [ ]:
df = pd.read_csv('rmse_prediction_results_cubic.csv')
results = analyze_regression_models(df)
print("\nPerformance Summary:")
print(results['performance_summary'])
print("\nStatistical Tests vs Best Model:")
print(results['statistical_tests'])
print("\nWin Ratio:")
print(results['win_ratio'].sort_values(ascending=False))